In [5]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder


def add_realistic_noise(df, feature_cols, noise_level=0.05, random_state=42):
    np.random.seed(random_state)
    df_noisy = df.copy()

    for col in feature_cols:
        multiplicative_noise = np.random.uniform(
            -noise_level,
            noise_level,
            size=len(df_noisy)
        )

        gaussian_noise = np.random.normal(
            loc=0,
            scale=df_noisy[col].std() * 0.03,
            size=len(df_noisy)
        )

        df_noisy[col] = df_noisy[col] + (df_noisy[col] * multiplicative_noise) + gaussian_noise

    # Keep values physically valid
    df_noisy["vibration_mm_s"] = df_noisy["vibration_mm_s"].clip(0.1, 6.0)
    df_noisy["motor_current_A"] = df_noisy["motor_current_A"].clip(1, 28)
    df_noisy["bearing_temperature_C"] = df_noisy["bearing_temperature_C"].clip(20, 95)
    df_noisy["suction_pressure_bar"] = df_noisy["suction_pressure_bar"].clip(0, 2.0)
    df_noisy["discharge_pressure_bar"] = df_noisy["discharge_pressure_bar"].clip(0, 7.0)
    df_noisy["flow_rate_lpm"] = df_noisy["flow_rate_lpm"].clip(0, 350)

    return df_noisy


def train_pump_failure_model(input_file, model_output="pump_failure_rf_model.pkl"):
    df = pd.read_csv(input_file)

    target_col = "active_failure"

    feature_cols = [
        "vibration_mm_s",
        "motor_current_A",
        "bearing_temperature_C",
        "suction_pressure_bar",
        "discharge_pressure_bar",
        "flow_rate_lpm",
    ]

    df = df.dropna(subset=feature_cols + [target_col])

    print("Class distribution:")
    print(df[target_col].value_counts())

    # Add noise to make synthetic data more realistic
    df = add_realistic_noise(df, feature_cols, noise_level=0.06)

    X = df[feature_cols]
    y = df[target_col]

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y_encoded,
        test_size=0.30,
        random_state=42,
        stratify=y_encoded
    )

    model = RandomForestClassifier(
        n_estimators=400,
        max_depth=12,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    print("\nTest Results:")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
    print("Weighted F1:", f1_score(y_test, y_pred, average="weighted"))

    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            y_pred,
            target_names=label_encoder.classes_
        )
    )

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(
        model,
        X,
        y_encoded,
        cv=cv,
        scoring="f1_macro",
        n_jobs=-1
    )

    print("\n5-Fold Macro F1 Scores:", cv_scores)
    print("Average CV Macro F1:", cv_scores.mean())

    feature_importance = pd.DataFrame({
        "feature": feature_cols,
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=False)

    print("\nFeature Importance:")
    print(feature_importance)

    feature_importance.to_csv(
        "/home/santhosh/AMDA/models/PUMP_feature_importance.csv",
        index=False
    )

    plt.figure(figsize=(8, 5))
    plt.barh(feature_importance["feature"], feature_importance["importance"])
    plt.gca().invert_yaxis()
    plt.title("Pump Failure Model - Feature Importance")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.savefig("/home/santhosh/AMDA/models/PUMP_feature_importance.png")
    plt.close()

    artifact = {
        "model": model,
        "label_encoder": label_encoder,
        "feature_cols": feature_cols,
        "classes": list(label_encoder.classes_),
    }

    joblib.dump(artifact, model_output)
    print(f"\nModel saved to: {model_output}")

    return model, label_encoder, feature_importance


model, label_encoder, feature_importance = train_pump_failure_model(
    "/home/santhosh/AMDA/datasets/PUMP_03_dataset.csv",
    "/home/santhosh/AMDA/models/PUMP/PUMP_failure_model.pkl"
)

Class distribution:
active_failure
none               1024
cavitation         1009
bearing_failure    1001
clogging            999
seal_leakage        997
impeller_damage     986
dry_run             984
Name: count, dtype: int64

Test Results:
Accuracy: 0.9038095238095238
Macro F1: 0.9033524187513798
Weighted F1: 0.9034218978213528

Classification Report:
                 precision    recall  f1-score   support

bearing_failure       0.90      0.86      0.88       300
     cavitation       0.92      0.89      0.91       303
       clogging       0.96      0.97      0.97       300
        dry_run       0.97      0.98      0.98       295
impeller_damage       0.84      0.82      0.83       296
           none       0.90      0.95      0.92       307
   seal_leakage       0.83      0.84      0.84       299

       accuracy                           0.90      2100
      macro avg       0.90      0.90      0.90      2100
   weighted avg       0.90      0.90      0.90      2100


Confusion M

## **Testing model against custom Test Cases**

In [6]:
import pandas as pd
import joblib


MODEL_PATH = "/home/santhosh/AMDA/models/PUMP/PUMP_failure_model.pkl"

artifact = joblib.load(MODEL_PATH)

model = artifact["model"]
label_encoder = artifact["label_encoder"]
feature_cols = artifact["feature_cols"]


test_cases = [
    {
        "case": "Healthy baseline",
        "expected": "none",
        "vibration_mm_s": 1.25,
        "motor_current_A": 12.0,
        "bearing_temperature_C": 54.0,
        "suction_pressure_bar": 1.10,
        "discharge_pressure_bar": 4.20,
        "flow_rate_lpm": 220.0,
    },
    {
        "case": "Healthy high load",
        "expected": "none",
        "vibration_mm_s": 1.55,
        "motor_current_A": 13.8,
        "bearing_temperature_C": 58.0,
        "suction_pressure_bar": 1.02,
        "discharge_pressure_bar": 4.45,
        "flow_rate_lpm": 235.0,
    },
    {
        "case": "Bearing failure early",
        "expected": "bearing_failure",
        "vibration_mm_s": 2.25,
        "motor_current_A": 14.2,
        "bearing_temperature_C": 67.0,
        "suction_pressure_bar": 1.05,
        "discharge_pressure_bar": 4.10,
        "flow_rate_lpm": 210.0,
    },
    {
        "case": "Bearing failure advanced",
        "expected": "bearing_failure",
        "vibration_mm_s": 3.35,
        "motor_current_A": 15.8,
        "bearing_temperature_C": 79.0,
        "suction_pressure_bar": 1.00,
        "discharge_pressure_bar": 4.00,
        "flow_rate_lpm": 200.0,
    },
    {
        "case": "Cavitation",
        "expected": "cavitation",
        "vibration_mm_s": 3.2,
        "motor_current_A": 13.8,
        "bearing_temperature_C": 61.0,
        "suction_pressure_bar": 0.55,
        "discharge_pressure_bar": 2.80,
        "flow_rate_lpm": 145.0,
    },
    {
        "case": "Severe cavitation",
        "expected": "cavitation",
        "vibration_mm_s": 4.6,
        "motor_current_A": 14.8,
        "bearing_temperature_C": 64.0,
        "suction_pressure_bar": 0.28,
        "discharge_pressure_bar": 2.30,
        "flow_rate_lpm": 105.0,
    },
    {
        "case": "Seal leakage",
        "expected": "seal_leakage",
        "vibration_mm_s": 1.75,
        "motor_current_A": 13.2,
        "bearing_temperature_C": 57.0,
        "suction_pressure_bar": 0.98,
        "discharge_pressure_bar": 3.10,
        "flow_rate_lpm": 175.0,
    },
    {
        "case": "Severe seal leakage",
        "expected": "seal_leakage",
        "vibration_mm_s": 2.10,
        "motor_current_A": 14.5,
        "bearing_temperature_C": 61.0,
        "suction_pressure_bar": 0.92,
        "discharge_pressure_bar": 2.55,
        "flow_rate_lpm": 145.0,
    },
    {
        "case": "Impeller damage",
        "expected": "impeller_damage",
        "vibration_mm_s": 2.45,
        "motor_current_A": 15.6,
        "bearing_temperature_C": 64.0,
        "suction_pressure_bar": 0.90,
        "discharge_pressure_bar": 2.35,
        "flow_rate_lpm": 125.0,
    },
    {
        "case": "Severe impeller damage",
        "expected": "impeller_damage",
        "vibration_mm_s": 3.05,
        "motor_current_A": 17.2,
        "bearing_temperature_C": 68.0,
        "suction_pressure_bar": 0.82,
        "discharge_pressure_bar": 1.75,
        "flow_rate_lpm": 95.0,
    },
    {
        "case": "Clogging",
        "expected": "clogging",
        "vibration_mm_s": 2.25,
        "motor_current_A": 18.5,
        "bearing_temperature_C": 68.0,
        "suction_pressure_bar": 0.72,
        "discharge_pressure_bar": 5.60,
        "flow_rate_lpm": 105.0,
    },
    {
        "case": "Severe clogging",
        "expected": "clogging",
        "vibration_mm_s": 2.75,
        "motor_current_A": 22.0,
        "bearing_temperature_C": 74.0,
        "suction_pressure_bar": 0.55,
        "discharge_pressure_bar": 6.45,
        "flow_rate_lpm": 70.0,
    },
    {
        "case": "Dry run",
        "expected": "dry_run",
        "vibration_mm_s": 2.75,
        "motor_current_A": 10.5,
        "bearing_temperature_C": 78.0,
        "suction_pressure_bar": 0.20,
        "discharge_pressure_bar": 0.90,
        "flow_rate_lpm": 35.0,
    },
    {
        "case": "Severe dry run",
        "expected": "dry_run",
        "vibration_mm_s": 4.0,
        "motor_current_A": 9.0,
        "bearing_temperature_C": 90.0,
        "suction_pressure_bar": 0.05,
        "discharge_pressure_bar": 0.30,
        "flow_rate_lpm": 5.0,
    },
    {
        "case": "Borderline seal leakage vs impeller damage",
        "expected": "impeller_damage",
        "vibration_mm_s": 2.20,
        "motor_current_A": 15.0,
        "bearing_temperature_C": 62.0,
        "suction_pressure_bar": 0.90,
        "discharge_pressure_bar": 2.55,
        "flow_rate_lpm": 135.0,
    },
    {
        "case": "Borderline cavitation vs impeller damage",
        "expected": "cavitation",
        "vibration_mm_s": 3.00,
        "motor_current_A": 14.5,
        "bearing_temperature_C": 62.0,
        "suction_pressure_bar": 0.65,
        "discharge_pressure_bar": 2.70,
        "flow_rate_lpm": 130.0,
    },
    {
        "case": "Borderline bearing failure vs healthy",
        "expected": "bearing_failure",
        "vibration_mm_s": 1.95,
        "motor_current_A": 13.5,
        "bearing_temperature_C": 63.0,
        "suction_pressure_bar": 1.05,
        "discharge_pressure_bar": 4.05,
        "flow_rate_lpm": 215.0,
    },
]


df_test = pd.DataFrame(test_cases)

X_test = df_test[feature_cols]

pred_encoded = model.predict(X_test)
pred_labels = label_encoder.inverse_transform(pred_encoded)
probabilities = model.predict_proba(X_test)

df_test["predicted"] = pred_labels
df_test["confidence"] = probabilities.max(axis=1).round(4)
df_test["correct"] = df_test["expected"] == df_test["predicted"]

top2_indices = probabilities.argsort(axis=1)[:, -2:][:, ::-1]

df_test["top_1"] = [
    label_encoder.classes_[idxs[0]] for idxs in top2_indices
]

df_test["top_1_confidence"] = [
    round(probabilities[i][idxs[0]], 4)
    for i, idxs in enumerate(top2_indices)
]

df_test["top_2"] = [
    label_encoder.classes_[idxs[1]] for idxs in top2_indices
]

df_test["top_2_confidence"] = [
    round(probabilities[i][idxs[1]], 4)
    for i, idxs in enumerate(top2_indices)
]

print("\nPUMP_03 Scenario Test Results:")
print(
    df_test[
        [
            "case",
            "expected",
            "predicted",
            "confidence",
            "top_1",
            "top_1_confidence",
            "top_2",
            "top_2_confidence",
            "correct",
        ]
    ]
)

accuracy = df_test["correct"].mean()
print(f"\nScenario Test Accuracy: {accuracy:.2%}")

output_path = "/home/santhosh/AMDA/models/PUMP/PUMP_manual_scenario_tests.csv"
df_test.to_csv(output_path, index=False)

print("\nSaved test results to:")
print(output_path)


PUMP_03 Scenario Test Results:
                                          case         expected  \
0                             Healthy baseline             none   
1                            Healthy high load             none   
2                        Bearing failure early  bearing_failure   
3                     Bearing failure advanced  bearing_failure   
4                                   Cavitation       cavitation   
5                            Severe cavitation       cavitation   
6                                 Seal leakage     seal_leakage   
7                          Severe seal leakage     seal_leakage   
8                              Impeller damage  impeller_damage   
9                       Severe impeller damage  impeller_damage   
10                                    Clogging         clogging   
11                             Severe clogging         clogging   
12                                     Dry run          dry_run   
13                            